# ED Eval Sidecar v4.1 — Ops-realistic + aligned artifacts
This notebook matches the last successful run’s ACTIONS/FEATURES and bakes in fixes:
- Higher volume, normalized continuous features
- Crisp mutually-exclusive labels
- Train-time boost for REQUEST_* so pages aren’t invisible to the model
- Stricter OPS budgets & recall floors
- Tighter capacity freshness + sparser updates for realistic policy blocks
- PR curves, confusion matrix, pager-load summary
- Artifact filenames aligned with the main project

In [1]:

import os, json, math, random, numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

ARTS = Path("artifacts"); ARTS.mkdir(exist_ok=True, parents=True)

SEED = 4242
np.random.seed(SEED); random.seed(SEED)

ACTIONS = ['NO_OP','ORDER_ECG','PERFORM_FAST','ORDER_LABS','ORDER_XR','ORDER_CT','REQUEST_CONSULT','REQUEST_BED']
N_ACTIONS = len(ACTIONS)

FEATURE_NAMES = ['minute_of_day','cap_stale','ems','consult_delay_min','syn_chest_pain','syn_polytrauma','syn_neuro_deficit',
                 'syn_other','ecg_hint','fast_hint','ct_hint','ems_prealert','risk_score']
N_FEATURES = len(FEATURE_NAMES)

# Volume
N_DAYS = 7
LAMBDA_SCALE = 2.5

# Priors
PREV = {'syn_chest_pain':0.22,'syn_polytrauma':0.06,'syn_neuro_deficit':0.08,'syn_other':0.64}
LAMBDA_BY_HOUR = {0:1.0,1:1.0,2:0.9,3:0.9,4:0.9,5:1.1,6:1.6,7:2.2,8:3.3,9:3.6,10:3.3,11:3.1,
                  12:3.0,13:3.0,14:3.0,15:3.2,16:3.5,17:3.7,18:3.0,19:2.5,20:2.0,21:1.8,22:1.4,23:1.2}
SIM_MINUTES = N_DAYS*24*60

# Recall floors (critical actions higher)
RECALL_FLOOR = {'ORDER_ECG':0.85,'ORDER_CT':0.85,'PERFORM_FAST':0.85,'ORDER_LABS':0.70,'ORDER_XR':0.65}
# Pager budgets per hour (strict)
FP_BUDGET_PER_H = {"REQUEST_CONSULT": 0.05, "REQUEST_BED": 0.02}
# Validation minima for rare pages
VAL_MIN = {"REQUEST_CONSULT": 20, "REQUEST_BED": 20}

print("ACTIONS:", ACTIONS)
print("FEATURE_NAMES:", FEATURE_NAMES)


ACTIONS: ['NO_OP', 'ORDER_ECG', 'PERFORM_FAST', 'ORDER_LABS', 'ORDER_XR', 'ORDER_CT', 'REQUEST_CONSULT', 'REQUEST_BED']
FEATURE_NAMES: ['minute_of_day', 'cap_stale', 'ems', 'consult_delay_min', 'syn_chest_pain', 'syn_polytrauma', 'syn_neuro_deficit', 'syn_other', 'ecg_hint', 'fast_hint', 'ct_hint', 'ems_prealert', 'risk_score']


In [2]:

def nonhom_poisson_arrivals(total_minutes, lam_by_hour, scale=1.0):
    out=[]
    for m in range(total_minutes):
        h=(m//60)%24; lam=lam_by_hour.get(h,1.0)*scale/60.0
        if np.random.rand()<lam: out.append(m)
    return out

def one_hot_syndrome():
    r=np.random.rand()
    if r<PREV['syn_chest_pain']: return 1,0,0,0
    r-=PREV['syn_chest_pain']
    if r<PREV['syn_polytrauma']: return 0,1,0,0
    r-=PREV['syn_polytrauma']
    if r<PREV['syn_neuro_deficit']: return 0,0,1,0
    return 0,0,0,1

def gen_dataset(total_minutes=SIM_MINUTES, scale=LAMBDA_SCALE):
    arr=nonhom_poisson_arrivals(total_minutes, LAMBDA_BY_HOUR, scale)
    X=[]; y=[]; ts=[]
    for idx, t in enumerate(arr):
        ems = int(np.random.rand()<0.55)
        ems_prealert = int(ems and (np.random.rand()<0.35))
        cap_stale = int(np.random.rand()<0.25)
        s_cp, s_poly, s_neuro, s_other = one_hot_syndrome()
        ecg_hint = int(s_cp or (np.random.rand()<0.15))
        fast_hint = int(s_poly or (np.random.rand()<0.10))
        ct_hint = int((s_poly or s_neuro) or (np.random.rand()<0.10))
        base_delay = np.random.normal(20, 8)  # minutes
        consult_delay_min_raw = max(0.0, base_delay - 6*ems_prealert + 5*cap_stale)
        risk_raw = np.clip(0.15 + 0.35*s_poly + 0.35*s_neuro + 0.10*ems + 0.10*ems_prealert, 0, 1)
        minute_of_day_raw = t % 1440

        # normalized/warped
        minute_of_day = minute_of_day_raw/1440.0
        consult_delay_min = np.log1p(consult_delay_min_raw/10.0)
        risk = risk_raw

        feats = [minute_of_day, cap_stale, ems, consult_delay_min, s_cp, s_poly, s_neuro,
                 s_other, ecg_hint, fast_hint, ct_hint, ems_prealert, risk]

        # --- Crisp mutually-exclusive label rules ---
        is_ct_path   = (ct_hint==1) and (s_neuro==1 or s_poly==1)
        is_cp_path   = (s_cp==1 and ecg_hint==1)
        is_sepsisish = (risk>0.55 and ems==1)

        if is_ct_path:
            base = np.random.choice(['ORDER_CT','PERFORM_FAST'], p=[0.85,0.15])
        elif is_cp_path:
            base = 'ORDER_ECG'
        elif is_sepsisish:
            base = 'ORDER_LABS'
        elif (s_other==1 and fast_hint==0 and np.random.rand()<0.25):
            base = 'ORDER_XR'
        else:
            base = 'NO_OP'

        # Escalation AFTER base decision
        escalate = (base in ['ORDER_CT','PERFORM_FAST','ORDER_LABS']) and (risk>0.65) and (np.random.rand()<0.28)
        if escalate:
            base = np.random.choice(['REQUEST_CONSULT','REQUEST_BED'], p=[0.75,0.25])

        label = base
        X.append(feats); y.append(ACTIONS.index(label)); ts.append(t)
    X=np.array(X, dtype=np.float32); y=np.array(y, dtype=np.int64); ts=np.array(ts, dtype=np.int64)
    # short sequences for GRU
    T=5
    X_seq = np.stack([X + 0.01*np.random.randn(*X.shape) for _ in range(T)], axis=1).astype(np.float32)
    return X_seq, y, ts

X, y, ts = gen_dataset()
print('Total patients:', len(X))


Total patients: 971


In [3]:

def split_idx(N, val_frac=0.17):
    idx=np.arange(N); np.random.shuffle(idx)
    n_val=max(1,int(N*val_frac))
    return np.array(sorted(idx[n_val:])), np.array(sorted(idx[:n_val]))

train_idx, val_idx = split_idx(len(y), 0.17)

def ensure_val_min(X, y, ts, train_idx, val_idx):
    Xtr, ytr, ttr = X[train_idx], y[train_idx], ts[train_idx]
    Xv, yv, tv = X[val_idx], y[val_idx], ts[val_idx]
    name_to_id = {n:i for i,n in enumerate(ACTIONS)}
    for name, req in VAL_MIN.items():
        cid = name_to_id[name]
        have = int((yv==cid).sum())
        need = req - have
        if need>0:
            src = Xtr[ytr==cid]
            if len(src)==0: continue
            aug = np.repeat(src[:1], repeats=need, axis=0) + 0.01*np.random.randn(need, *src.shape[1:])
            Xv = np.concatenate([Xv, aug], axis=0)
            yv = np.concatenate([yv, np.full(need, cid, dtype=np.int64)], axis=0)
            tv = np.concatenate([tv, np.random.choice(tv, size=need, replace=True)], axis=0)
    return Xtr, ytr, ttr, Xv, yv, tv

Xtr, ytr, ttr, Xv, yv, tv = ensure_val_min(X, y, ts, train_idx, val_idx)

# --- Ensure training has some REQUEST_* exposure ---
name_to_id = {n:i for i,n in enumerate(ACTIONS)}
def boost_train_min(Xtr, ytr, target_min=40):
    aug_X = []
    aug_y = []
    for name in ['REQUEST_CONSULT','REQUEST_BED']:
        cid = name_to_id[name]
        have = int((ytr==cid).sum())
        need = max(0, target_min - have)
        if need>0:
            src = Xtr[ytr==cid]
            if len(src)==0:
                proxy_ids = np.where((ytr==name_to_id['ORDER_CT']) | (ytr==name_to_id['ORDER_LABS']))[0]
                if len(proxy_ids)>0:
                    src = Xtr[proxy_ids]
            if len(src)>0:
                noise = 0.01*np.random.randn(need, *src.shape[1:])
                aug_X.append(np.repeat(src[:1], repeats=need, axis=0) + noise)
                aug_y.append(np.full(need, cid, dtype=np.int64))
    if aug_X:
        Xtr = np.concatenate([Xtr, *aug_X], axis=0)
        ytr = np.concatenate([ytr, *aug_y], axis=0)
    return Xtr, ytr

Xtr, ytr = boost_train_min(Xtr, ytr, target_min=40)

from collections import Counter
print('Train/Val:', Xtr.shape, Xv.shape)
print('Train classes (boosted):', Counter(ytr))
print('Val classes:  ', Counter(yv))


Train/Val: (879, 5, 13) (184, 5, 13)
Train classes (boosted): Counter({0: 388, 1: 201, 4: 107, 5: 88, 6: 40, 7: 40, 2: 15})
Val classes:   Counter({0: 75, 1: 43, 4: 21, 6: 20, 5: 19, 2: 5, 7: 1})


In [4]:

import torch, torch.nn as nn, torch.nn.functional as F
device = torch.device("cpu")

class GRUHead(nn.Module):
    def __init__(self, in_f, hidden=128, n_actions=8, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(in_f, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_actions)
    def forward(self, x):
        out,_ = self.gru(x); h=self.drop(out[:,-1,:]); return self.fc(h)

def class_weights(y, n_classes):
    cnt = np.bincount(y, minlength=n_classes).astype(float)
    inv = 1.0/np.sqrt(cnt+1e-6)
    w = inv / inv.sum() * n_classes
    return torch.tensor(w, dtype=torch.float32)

model = GRUHead(N_FEATURES, hidden=128, n_actions=len(ACTIONS), dropout=0.2).to(device)
weights = class_weights(ytr, len(ACTIONS)).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW(model.parameters(), lr=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=35)

def to_t(x): return torch.tensor(x, dtype=torch.float32).to(device)
def to_y(x): return torch.tensor(x, dtype=torch.long).to(device)

B=256
def batches(X,y,bs=B):
    N=len(y); idx=np.arange(N); np.random.shuffle(idx)
    for i in range(0,N,bs):
        b=idx[i:i+bs]; yield X[b], y[b]

for ep in range(1,36):
    model.train(); tl=0.0; tn=0
    for xb,yb in batches(Xtr,ytr,B):
        xb_t, yb_t = to_t(xb), to_y(yb)
        opt.zero_grad(); loss=criterion(model(xb_t), yb_t); loss.backward(); opt.step()
        tl += float(loss.detach())*len(yb); tn += len(yb)
    model.eval()
    with torch.no_grad(): vl=float(criterion(model(to_t(Xv)), to_y(yv)).detach())
    sched.step()
    if ep%5==0 or ep in [1,2,3]: print(f"Epoch {ep:02d}: train={tl/tn:.4f} val={vl:.4f}")


Epoch 01: train=2.0797 val=2.0361
Epoch 02: train=2.0313 val=1.9883
Epoch 03: train=1.9821 val=1.9427
Epoch 05: train=1.8805 val=1.8463
Epoch 10: train=1.5736 val=1.5527
Epoch 15: train=1.1826 val=1.1352
Epoch 20: train=0.8950 val=0.8699
Epoch 25: train=0.7904 val=0.7900
Epoch 30: train=0.7615 val=0.7701
Epoch 35: train=0.7664 val=0.7669


In [5]:

with torch.no_grad():
    val_logits = model(to_t(Xv)).cpu().numpy()

def softmax(z,T=1.0,axis=1):
    zT=z/max(T,1e-6); zT-=zT.max(axis=axis,keepdims=True); e=np.exp(zT); return e/e.sum(axis=axis,keepdims=True)
def nll(p,y): 
    idx=np.arange(len(y)); return -np.log(np.clip(p[idx,y],1e-9,1)).mean()

Ts=np.linspace(0.6,1.8,25); bestT, best=1.0, 1e9
for T in Ts:
    p=softmax(val_logits,T=T); cur=nll(p,yv)
    if cur<best: best, bestT = cur, T
print(f"[Calib] Best T={bestT:.3f}")
probs=softmax(val_logits,T=bestT)

# Save val cache
np.savez(ARTS/'val_cache_sidecar_v4_1.npz', probs=probs.astype('float32'), y_true=yv.astype('int64'),
         action_names=np.array(ACTIONS, dtype=object), ts=tv.astype('int64'), T=bestT,
         feature_names=np.array(FEATURE_NAMES, dtype=object))

# Threshold pickers
def pick_tau_cost(y_true_bin, y_prob, fp_cost=1.0, fn_cost=5.0):
    taus=np.linspace(0,1,101); best=None
    for t in taus:
        yp=(y_prob>=t).astype(int)
        tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())
        prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9)
        f1=2*prec*rec/(prec+rec+1e-9); cost=fp_cost*fp+fn_cost*fn
        if best is None or cost<best[0]: best=(cost,t,prec,rec,f1,tp,fp,fn)
    c,t,prec,rec,f1,tp,fp,fn=best
    return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
            "cost":float(c),"tp":tp,"fp":fp,"fn":fn}

def pick_tau_ops(y_true_bin, y_prob, ts_min, name):
    taus=np.linspace(0,1,101); hours=(ts_min.max()-ts_min.min()+1)/60.0; best=None
    recall_floor = RECALL_FLOOR.get(name, None)
    fp_budget_per_h = FP_BUDGET_PER_H.get(name, None)
    for t in taus:
        yp=(y_prob>=t).astype(int)
        tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())
        prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9)
        f1=2*prec*rec/(prec+rec+1e-9); fp_per_h=fp/max(hours,1e-6)
        if recall_floor is not None and rec<recall_floor: continue
        if fp_budget_per_h is not None and fp_per_h>fp_budget_per_h: continue
        score=f1
        if best is None or score>best[0]: best=(score,t,prec,rec,f1,tp,fp,fn,fp_per_h)
    if best is None:
        t=0.5; yp=(y_prob>=t).astype(int)
        tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())
        prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
        fp_per_h=fp/((ts_min.max()-ts_min.min()+1)/60.0)
        return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
                "tp":tp,"fp":fp,"fn":fn,"fp_per_h":float(fp_per_h),"status":"fallback"}
    score,t,prec,rec,f1,tp,fp,fn,fp_per_h=best
    return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
            "tp":tp,"fp":fp,"fn":fn,"fp_per_h":float(fp_per_h)}

thr_cost={}; thr_ops={}; notes={}
for ci,name in enumerate(ACTIONS):
    yb=(yv==ci).astype(int)
    thr_cost[name]=pick_tau_cost(yb, probs[:,ci])
    thr_ops[name]=pick_tau_ops(yb, probs[:,ci], tv, name)
    notes[name]={'recall_floor':RECALL_FLOOR.get(name), 'fp_budget_per_h':FP_BUDGET_PER_H.get(name)}

with open(ARTS/'thresholds.json','w') as f: json.dump(thr_cost, f, indent=2)
with open(ARTS/'thresholds_ops.json','w') as f: json.dump(thr_ops, f, indent=2)
with open(ARTS/'thresholds_ops_notes.json','w') as f: json.dump(notes, f, indent=2)

import shutil
shutil.copyfile(ARTS/'thresholds.json', ARTS/'per_class_thresholds_cost.json')
shutil.copyfile(ARTS/'thresholds_ops.json', ARTS/'per_class_thresholds_ops.json')

# PR curves (critical)
critical = ['ORDER_CT','PERFORM_FAST','ORDER_ECG']
plt.figure()
for name in critical:
    ci = ACTIONS.index(name)
    yb = (yv==ci).astype(int)
    prec, rec, thr = precision_recall_curve(yb, probs[:,ci])
    plt.plot(rec, prec, label=name)
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("PR Curves (critical actions)"); plt.legend()
plt.tight_layout(); plt.savefig(ARTS/'pr_curves_critical.png', dpi=160); plt.close()

# Confusion matrix
y_pred = probs.argmax(axis=1)
cm = np.zeros((len(ACTIONS), len(ACTIONS)), dtype=int)
for t,p in zip(yv, y_pred): cm[t,p]+=1
plt.figure()
plt.imshow(cm, interpolation="nearest")
plt.title("Val Confusion Matrix"); plt.xlabel("Pred"); plt.ylabel("True"); plt.colorbar()
plt.tight_layout(); plt.savefig(ARTS/'cm_sidecar.png', dpi=160); plt.close()


[Calib] Best T=0.600


In [6]:

def capacity_fresh(ts_now, last_update_ts, max_age_min=10):
    if last_update_ts is None: return False
    return (ts_now - last_update_ts) <= max_age_min

def simulate_capacity_updates(total_minutes=SIM_MINUTES):
    updates=[]; t=0
    while t<total_minutes:
        updates.append(t); t += int(np.random.uniform(12,28))
    return updates

def policy_blocks(state, action_name, pkt):
    if action_name in ['REQUEST_CONSULT','REQUEST_BED']:
        if not pkt.get('packet_complete', True): return False, 'packet_incomplete'
        if not pkt.get('trauma_cleared', True):  return False, 'ownership'
        if not capacity_fresh(pkt['ts_now'], state.get('last_capacity_update_ts', None)): return False, 'capacity_stale'
        key=(pkt['patient_id'], action_name, pkt.get('target_service','IM'))
        if key in state['sent_packets']: return False, 'duplicate'
    return True, None

def make_packet(patient_id, ts_now):
    return {'patient_id':patient_id,'ts_now':ts_now,'target_service':'IM','packet_complete':True,'trauma_cleared':True}

import torch
def replay(thresholds, Tcal):
    logs={'proposals':[], 'gate_blocks':[], 'policy_blocks':[], 'approvals':[]}
    state={'last_capacity_update_ts':None, 'sent_packets':set()}
    caps=set(simulate_capacity_updates())
    order=np.argsort(ts)
    for i in order:
        tnow=int(ts[i])
        if tnow in caps: state['last_capacity_update_ts']=tnow
        with torch.no_grad():
            pr=torch.softmax(model(to_t(X[i:i+1]))/Tcal, dim=-1).cpu().numpy()[0]
        picks=[(ACTIONS[c], float(pr[c])) for c in range(len(ACTIONS)) if pr[c]>=thresholds.get(ACTIONS[c],{}).get('tau',0.5)]
        if not picks: continue
        picks_sorted=sorted(picks, key=lambda x: (x[0]=='NO_OP', -x[1]))
        name,score=picks_sorted[0]; tau=thresholds.get(name,{}).get('tau',0.5)
        if score<tau:
            logs['gate_blocks'].append({'action':name,'p':score,'tau':tau,'reason':'below_tau'}); continue
        if name in ['REQUEST_CONSULT','REQUEST_BED']:
            pkt=make_packet(int(i), tnow)
            ok,reason=policy_blocks(state, name, pkt)
            if not ok: logs['policy_blocks'].append({'action':name,'reason':reason}); continue
            state['sent_packets'].add((pkt['patient_id'], name, pkt['target_service']))
        logs['proposals'].append({'idx':int(i),'action':name,'p':score,'ts':tnow})
        if name in ['REQUEST_CONSULT','REQUEST_BED']:
            if np.random.rand()<0.7: logs['approvals'].append({'idx':int(i),'action':name,'ts':tnow})
        else:
            logs['approvals'].append({'idx':int(i),'action':name,'ts':tnow})
    return logs

dat=np.load(ARTS/'val_cache_sidecar_v4_1.npz', allow_pickle=True)
Tcal=float(dat['T'])
thr_cost=json.load(open(ARTS/'per_class_thresholds_cost.json'))
thr_ops=json.load(open(ARTS/'per_class_thresholds_ops.json'))

logs_cost=replay(thr_cost, Tcal)
logs_ops=replay(thr_ops, Tcal)

def summarize(logs, label):
    props=len(logs['proposals']); appr=len(logs['approvals'])
    print(f"{label}: proposals={props} approvals={appr} approve_rate={appr/max(1,props):.2f} "
          f"gate_blocks={len(logs['gate_blocks'])} policy_blocks={len(logs['policy_blocks'])}")

from collections import Counter
summarize(logs_cost, "Replay (PR/cost)")
print("Policy blocks (PR):", Counter([b['reason'] for b in logs_cost['policy_blocks']]))
summarize(logs_ops,  "Replay (ops)")
print("Policy blocks (OPS):", Counter([b['reason'] for b in logs_ops['policy_blocks']]))

def pager_load(logs):
    pages = [p for p in logs["proposals"] if p["action"] in ["REQUEST_CONSULT","REQUEST_BED"]]
    if not pages: return {"pages":0,"per_hour":0.0}
    t0 = min(p["ts"] for p in pages); t1 = max(p["ts"] for p in pages)
    hours = max(1.0, (t1 - t0)/60.0)
    return {"pages":len(pages), "per_hour": round(len(pages)/hours, 3)}

print("Pager load (PR):", pager_load(logs_cost))
print("Pager load (OPS):", pager_load(logs_ops))

def kpis_from_logs(logs):
    props = logs["proposals"]; appr = logs["approvals"]
    approve_rate = len(appr)/max(1,len(props))
    saved = 0.0
    for p in props:
        if p["action"] in ["ORDER_ECG","ORDER_LABS"]: saved += 2.0
        elif p["action"] in ["ORDER_CT","PERFORM_FAST"]: saved += 5.0
        elif p["action"] in ["REQUEST_CONSULT","REQUEST_BED"]: saved += 4.0
    median_minutes_saved = saved / max(1,len(props))
    page_props = [p for p in props if p["action"] in ["REQUEST_CONSULT","REQUEST_BED"]]
    page_appr  = [a for a in appr  if a["action"] in ["REQUEST_CONSULT","REQUEST_BED"]]
    fpr = (len(page_props)-len(page_appr))/max(1,len(page_props))
    return {"proposals":len(props),"approvals":len(appr),"approve_rate":round(approve_rate,3),
            "false_page_rate":round(fpr,3),"median_minutes_saved":round(median_minutes_saved,2)}

kpi_report = {"PR_cost": kpis_from_logs(logs_cost), "OPS": kpis_from_logs(logs_ops)}
with open(ARTS/'kpi_report.json','w') as f: json.dump(kpi_report, f, indent=2)
print("KPI report:", kpi_report)


Replay (PR/cost): proposals=965 approvals=964 approve_rate=1.00 gate_blocks=0 policy_blocks=6
Policy blocks (PR): Counter({'capacity_stale': 6})
Replay (ops): proposals=964 approvals=963 approve_rate=1.00 gate_blocks=0 policy_blocks=7
Policy blocks (OPS): Counter({'capacity_stale': 7})
Pager load (PR): {'pages': 3, 'per_hour': 0.055}
Pager load (OPS): {'pages': 2, 'per_hour': 0.178}
KPI report: {'PR_cost': {'proposals': 965, 'approvals': 964, 'approve_rate': 0.999, 'false_page_rate': 0.333, 'median_minutes_saved': 1.18}, 'OPS': {'proposals': 964, 'approvals': 963, 'approve_rate': 0.999, 'false_page_rate': 0.5, 'median_minutes_saved': 1.17}}
